In [ ]:
# DAY 2: Feature Engineering
# This notebook creates advanced features for win prediction

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.append('../src')

# Import our functions
try:
    from feature_engineering import create_advanced_features, prepare_model_datasets, get_important_features
    print("Successfully imported feature engineering functions")
except ImportError as e:
    print(f"Import error: {e}")
    print("Creating dummy functions for now...")
    
    # Create dummy functions if import fails
    def create_advanced_features(df, total_overs=20):
        print("Using dummy function - please implement the real one")
        return df
    
    def prepare_model_datasets(df, match_summary):
        return df[df['inning'] == 1], df[df['inning'] == 2]
    
    def get_important_features():
        return []

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8')

print("="*70)
print("DAY 2: ADVANCED FEATURE ENGINEERING")
print("="*70)

# ============================================================================
# PART 1: LOAD AND VERIFY DATA FROM DAY 1
# ============================================================================
print("\n" + "="*70)
print("PART 1: LOADING DATA FROM DAY 1")
print("="*70)

try:
    # Load cleaned data from Day 1
    df = pd.read_csv('../data/processed/cleaned_data.csv')
    match_summary = pd.read_csv('../data/processed/match_summary.csv')
    
    print(f"✅ Loaded cleaned data: {df.shape}")
    print(f"✅ Loaded match summary: {match_summary.shape}")
    
    # Quick verification
    print(f"\n📊 Data Verification:")
    print(f"   Unique matches: {df['match_id'].nunique()}")
    print(f"   Total deliveries: {len(df)}")
    print(f"   Columns from Day 1: {list(df.columns)}")
    
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("Please run Day 1 code first!")
    exit()

# ============================================================================
# PART 2: ANALYZE MATCH SUMMARY
# ============================================================================
print("\n" + "="*70)
print("PART 2: MATCH OUTCOME ANALYSIS")
print("="*70)

print("\n🏏 MATCH SUMMARY STATISTICS:")
print("-" * 40)
print(f"Total matches: {len(match_summary)}")
print(f"Average 1st innings score: {match_summary['team1_score'].mean():.1f}")
print(f"Average 2nd innings score: {match_summary['team2_score'].mean():.1f}")

# Calculate chasing success
second_innings_wins = match_summary[match_summary['winner'] == match_summary['team2']]
chasing_success_rate = len(second_innings_wins) / len(match_summary) * 100
print(f"Chasing success rate: {chasing_success_rate:.1f}%")

# Win distribution
print(f"\n🏆 WIN DISTRIBUTION (Top 10 teams):")
top_teams = match_summary['winner'].value_counts().head(10)
for team, wins in top_teams.items():
    print(f"   {team}: {wins} wins")

# Visualize score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Score distribution
axes[0].hist(match_summary['team1_score'], bins=30, alpha=0.7, label='1st Innings', color='blue')
axes[0].hist(match_summary['team2_score'], bins=30, alpha=0.7, label='2nd Innings', color='orange')
axes[0].set_xlabel('Total Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Score Distribution by Innings')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Win margin
axes[1].hist(match_summary['win_by'], bins=30, alpha=0.7, color='green')
axes[1].set_xlabel('Win Margin (Runs/Wickets)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Win Margin Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/day2_match_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

# ============================================================================
# PART 3: CREATE ADVANCED FEATURES
# ============================================================================
print("\n" + "="*70)
print("PART 3: CREATING ADVANCED FEATURES")
print("="*70)

# Determine match format (T20 or ODI)
max_overs = df.groupby('match_id')['over'].max().max()
total_overs = 20 if max_overs <= 20 else 50
print(f"\n🎯 Match Format Detected: {'T20' if total_overs == 20 else 'ODI'} ({total_overs} overs)")

# Create advanced features
print("\n🔧 Creating advanced features...")
df_features = create_advanced_features(df, total_overs=total_overs)

print(f"\n✅ Feature Engineering Complete!")
print(f"   Original columns: {len(df.columns)}")
print(f"   New columns: {len(df_features.columns)}")
print(f"   Total columns now: {len(df_features.columns)}")

# Show new features
new_features = [col for col in df_features.columns if col not in df.columns]
print(f"\n📋 New Features Created ({len(new_features)}):")
for i, feature in enumerate(new_features[:20]):  # Show first 20
    print(f"   {i+1:2d}. {feature}")
if len(new_features) > 20:
    print(f"   ... and {len(new_features) - 20} more")

# ============================================================================
# PART 4: PREPARE MODEL DATASETS
# ============================================================================
print("\n" + "="*70)
print("PART 4: PREPARING MODEL DATASETS")
print("="*70)

first_innings_df, second_innings_df = prepare_model_datasets(df_features, match_summary)

print(f"\n📊 DATASET SUMMARY:")
print(f"   First Innings Dataset: {first_innings_df.shape}")
print(f"   Second Innings Dataset: {second_innings_df.shape}")

# Show sample of second innings data
print(f"\n👀 Sample of Second Innings Data (5 random rows):")
sample_cols = ['match_id', 'inning', 'over', 'ball', 'cumulative_runs', 
               'cumulative_wickets', 'target', 'runs_needed', 'required_run_rate',
               'batting_team_won']
sample = second_innings_df[sample_cols].sample(5, random_state=42)
print(sample.to_string(index=False))

# ============================================================================
# PART 5: FEATURE ANALYSIS
# ============================================================================
print("\n" + "="*70)
print("PART 5: FEATURE ANALYSIS")
print("="*70)

# Get important features
important_features = get_important_features()
print(f"\n📈 Important Features for Modeling ({len(important_features)}):")
for i in range(0, len(important_features), 5):
    print("   " + ", ".join(important_features[i:i+5]))

# Calculate correlation with target for second innings
if len(second_innings_df) > 0 and 'batting_team_won' in second_innings_df.columns:
    print(f"\n🔗 Feature Correlations with Win Probability:")
    
    # Check which features exist in our dataframe
    available_features = [f for f in important_features if f in second_innings_df.columns]
    
    correlations = {}
    for feature in available_features[:15]:  # Check first 15 features
        try:
            corr = second_innings_df[feature].corr(second_innings_df['batting_team_won'])
            correlations[feature] = corr
        except:
            continue
    
    # Sort by absolute correlation
    sorted_corrs = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    print("\n   Feature                   Correlation")
    print("   " + "-"*40)
    for feature, corr in sorted_corrs[:10]:
        print(f"   {feature:25} {corr:+.3f}")

# ============================================================================
# PART 6: SAVE PROCESSED DATA
# ============================================================================
print("\n" + "="*70)
print("PART 6: SAVING PROCESSED DATA")
print("="*70)

# Save all datasets
df_features.to_csv('../data/processed/data_with_advanced_features.csv', index=False)
first_innings_df.to_csv('../data/processed/first_innings_model_ready.csv', index=False)
second_innings_df.to_csv('../data/processed/second_innings_model_ready.csv', index=False)

print(f"\n💾 Data saved successfully!")
print(f"   📁 data_with_advanced_features.csv")
print(f"   📁 first_innings_model_ready.csv")
print(f"   📁 second_innings_model_ready.csv")

# ============================================================================
# PART 7: FINAL ANALYSIS AND VISUALIZATION
# ============================================================================
print("\n" + "="*70)
print("PART 7: FINAL ANALYSIS")
print("="*70)

# Create a visualization of key metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Runs needed vs Win probability
if 'runs_needed' in second_innings_df.columns:
    axes[0,0].scatter(second_innings_df['runs_needed'], 
                     second_innings_df['batting_team_won'], 
                     alpha=0.1, s=1)
    axes[0,0].set_xlabel('Runs Needed')
    axes[0,0].set_ylabel('Win (1) / Loss (0)')
    axes[0,0].set_title('Runs Needed vs Outcome')
    axes[0,0].grid(True, alpha=0.3)

# Plot 2: Required RR vs Win probability
if 'required_run_rate' in second_innings_df.columns:
    axes[0,1].scatter(second_innings_df['required_run_rate'], 
                     second_innings_df['batting_team_won'], 
                     alpha=0.1, s=1)
    axes[0,1].set_xlabel('Required Run Rate')
    axes[0,1].set_ylabel('Win (1) / Loss (0)')
    axes[0,1].set_title('Required RR vs Outcome')
    axes[0,1].grid(True, alpha=0.3)

# Plot 3: Wickets remaining vs Win probability
if 'wickets_remaining' in second_innings_df.columns:
    axes[0,2].scatter(second_innings_df['wickets_remaining'], 
                     second_innings_df['batting_team_won'], 
                     alpha=0.1, s=1)
    axes[0,2].set_xlabel('Wickets Remaining')
    axes[0,2].set_ylabel('Win (1) / Loss (0)')
    axes[0,2].set_title('Wickets vs Outcome')
    axes[0,2].grid(True, alpha=0.3)

# Plot 4: Phase-wise win rate
if 'phase' in second_innings_df.columns:
    phase_win_rate = second_innings_df.groupby('phase')['batting_team_won'].mean()
    axes[1,0].bar(phase_win_rate.index, phase_win_rate.values, color=['red', 'blue', 'green'])
    axes[1,0].set_xlabel('Match Phase')
    axes[1,0].set_ylabel('Win Rate')
    axes[1,0].set_title('Win Rate by Match Phase')
    axes[1,0].grid(True, alpha=0.3, axis='y')

# Plot 5: Over-by-over win probability
if 'over' in second_innings_df.columns:
    over_win_rate = second_innings_df.groupby('over')['batting_team_won'].mean()
    axes[1,1].plot(over_win_rate.index, over_win_rate.values, marker='o', linewidth=2)
    axes[1,1].set_xlabel('Over')
    axes[1,1].set_ylabel('Win Rate')
    axes[1,1].set_title('Win Rate by Over')
    axes[1,1].grid(True, alpha=0.3)
    axes[1,1].set_xticks(range(0, 21, 2))

# Plot 6: Resources remaining vs Win probability
if 'resources_remaining' in second_innings_df.columns:
    axes[1,2].scatter(second_innings_df['resources_remaining'], 
                     second_innings_df['batting_team_won'], 
                     alpha=0.1, s=1)
    axes[1,2].set_xlabel('Resources Remaining')
    axes[1,2].set_ylabel('Win (1) / Loss (0)')
    axes[1,2].set_title('Resources vs Outcome')
    axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/day2_feature_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

# ============================================================================
# PART 8: SUMMARY AND NEXT STEPS
# ============================================================================
print("\n" + "="*70)
print("DAY 2 COMPLETED SUCCESSFULLY! 🎉")
print("="*70)

print(f"\n📋 SUMMARY:")
print(f"   • Processed {len(df)} deliveries from {df['match_id'].nunique()} matches")
print(f"   • Created {len(new_features)} new features")
print(f"   • Prepared {len(first_innings_df)} first innings samples")
print(f"   • Prepared {len(second_innings_df)} second innings samples")
print(f"   • Overall chasing win rate: {chasing_success_rate:.1f}%")

print(f"\n🚀 NEXT STEPS (Day 3):")
print(f"   1. Build First Innings Score Predictor")
print(f"   2. Build Second Innings Win Probability Model")
print(f"   3. Implement Monte Carlo Simulation")

print(f"\n📁 FILES CREATED TODAY:")
print(f"   • notebooks/02_feature_engineering.ipynb")
print(f"   • src/feature_engineering.py (updated)")
print(f"   • data/processed/data_with_advanced_features.csv")
print(f"   • data/processed/first_innings_model_ready.csv")
print(f"   • data/processed/second_innings_model_ready.csv")
print(f"   • reports/figures/day2_*.png")

print(f"\n✅ Day 2 is complete! You're ready for model building tomorrow.")